In [ ]:
# !pip install unstructured
# !pip install unstructured-ingest
# !pip install "unstructured[pdf]"

In [1]:
# !apt-get update
# !pip install poppler-utils tesseract-ocr
# !pip install langchain-community
# !pip install langchain-classic
# !pip install sentence-transformers
# !pip install faiss-cpu  
# !pip install langchain langchain-core langchain-community langchain-ollama
# !pip install -U langchain-huggingface

In [5]:
%%time
from unstructured_ingest.pipeline.pipeline import Pipeline
from unstructured_ingest.interfaces import ProcessorConfig
from unstructured_ingest.processes.connectors.local import (
    LocalIndexerConfig, LocalDownloaderConfig,
    LocalConnectionConfig, LocalUploaderConfig
)
from unstructured_ingest.processes.partitioner import PartitionerConfig
from unstructured_ingest.processes.chunker import ChunkerConfig  #built-in chunking

Pipeline.from_configs(
    context=ProcessorConfig(
        tqdm=True,
        num_processes=2,       
        device="cuda"           
    ),
    indexer_config=LocalIndexerConfig(input_path="./pdfs"),     
    downloader_config=LocalDownloaderConfig(),
    source_connection_config=LocalConnectionConfig(),
    partitioner_config=PartitionerConfig(
        strategy="ocr_only", # efficernt parsing , extraction 
        languages=["eng"],
        additional_partition_args={
            "split_pdf_page": True,
            "split_pdf_concurrency_level": 15,
            "max_partition": 1500,
            "preserve_formatting": True,
            "infer_table_structure": True,
            "include_page_breaks": True,
        }
    ),
    
    chunker_config=ChunkerConfig(
         chunking_strategy="by_title",
         chunk_max_characters=1200,
         chunk_new_after_n_chars=1000,
         chunk_overlap=200,
         chunk_combine_text_under_n_chars=300,
         chunk_multipage_sections=True,
         chunk_include_orig_elements=True
    ),
    uploader_config=LocalUploaderConfig(output_dir="./parsed_pdfs")
).run()

Overriding of current TracerProvider is not allowed
2026-05-18 17:46:36,366 MainProcess INFO     created indexer with configs: {"input_path":"pdfs","recursive":false}, connection configs: {"access_config":"**********"}
2026-05-18 17:46:36,366 MainProcess INFO     created indexer with configs: {"input_path":"pdfs","recursive":false}, connection configs: {"access_config":"**********"}
2026-05-18 17:46:36,369 MainProcess INFO     Created download with configs: {"download_dir":null}, connection configs: {"access_config":"**********"}
2026-05-18 17:46:36,369 MainProcess INFO     Created download with configs: {"download_dir":null}, connection configs: {"access_config":"**********"}
2026-05-18 17:46:36,371 MainProcess INFO     created partition with configs: {"strategy":"ocr_only","ocr_languages":null,"encoding":null,"additional_partition_args":{"split_pdf_page":true,"split_pdf_concurrency_level":15,"max_partition":1500,"preserve_formatting":true,"infer_table_structure":true,"include_page_br

CPU times: total: 250 ms
Wall time: 10.9 s


***

## ***Embeddings*** ----------------------- 
### *Building FAISS AND VECTOR DATABASE*

***

In [6]:
%%time
import os
from pathlib import Path
from unstructured.staging.base import elements_from_json
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
docs = []
for f in Path("./parsed_pdfs").glob("*.json"):
    for el in elements_from_json(filename = str(f)):
        if not el.text:
            continue
        meta = el.metadata.to_dict()
        meta["is_table"] = type(el).__name__ == "Table"
        docs.append(Document(page_content = el.text, metadata = meta))

print({f"Total chunks :{len(docs)}"})
model = HuggingFaceEmbeddings(model_name = "intfloat/multilingual-e5-large")

db = FAISS.from_documents(
    docs,model
)
db.save_local("./db/vector_store")
print("Vector Database Saved !---")

{'Total chunks :354'}


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Database Saved !---
CPU times: total: 28min 33s
Wall time: 8min 5s


## Load Database

In [7]:
%%time
embeddings = model

db = FAISS.load_local(
    "./db/vector_store",
    embeddings,
    allow_dangerous_deserialization=True  # needed for .pkl
)
print(f"Loaded DB with {db.index.ntotal} vectors")

Loaded DB with 354 vectors
CPU times: total: 0 ns
Wall time: 70.2 ms


In [ ]:
%%time
query = "what is HArdness in glass and how it is measured"
results = db.similarity_search(query, k=3)

# results =  db.max_marginal_relevance_search(
#     query,
#     k=10,           # chunks returned
#     fetch_k=50,     # candidates to consider
#     lambda_mult=0.5 # 0 = max diversity, 1 = max relevance
# )

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source : {doc.metadata.get('filename', 'unknown')}")
    print(f"Page   : {doc.metadata.get('page_number', '?')}")
    # print(f"Table  : {doc.metadata.get('is_table', False)}")  //can be used when the parsing strategy is "hi_res"
    print(f"Text   : {doc.page_content[:300]}")

## LLM and Retriever

In [13]:
%%time
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA  
from langchain_core.prompts  import PromptTemplate

retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 16, "fetch_k": 40, "lambda_mult": 0.5}
)

llm = OllamaLLM(
    model="llama3.1:8b",   
    temperature=0.2,
    num_ctx=4096, #16384
)

CPU times: total: 31.2 ms
Wall time: 46.2 ms


## Prompt:Chain

In [14]:
%%time
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert scientific research assistant.

Your task is to answer the user's question using ONLY the provided context.

Rules:
1. Do NOT use external knowledge.
2. If the answer is not fully available in the context, explicitly say:
   "The provided context does not contain enough information to answer this question."
3. Be scientifically accurate and detailed.
4. Use clear technical explanations.
5. If equations are present:
   - reproduce them properly
   - explain the variables and meaning
6. If tables are present:
   - summarize the important findings
   - preserve numerical values when relevant
7. Structure the answer logically with headings when needed.
8. Do not invent citations, results, or experimental data.
9. Prefer concise precision over unnecessary verbosity.

Context:
---------------------
{context}
---------------------

Question:
{question}

Scientific Answer:
"""
)



qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

CPU times: total: 0 ns
Wall time: 3.01 ms


## test

In [15]:
%%time
# query = "Thermal physical parameters of ((Ti40Zr40Ni20)72Be28)100-xAlx BMGs"
query = "tell me about the document design strategies for stable glass"
result = qa.invoke({"query": query})

print(f"\n{'='*60}")
print(f"Query: {query}")
print(f"{'='*60}")
print(f"\nBOT : {result['result']}")

print(f"\n--- Sources ---{"-" * 60}")
for doc in result["source_documents"]:
    print(f"  File : {doc.metadata.get('filename', 'unknown')}")
    print(f"  Page : {doc.metadata.get('page_number', '?')}")
    print(f"  Text : {doc.page_content[:150]}\n")


Query: tell me about the document design strategies for stable glass

BOT : **Design Strategies for Stable Glasses**

The provided context discusses various design strategies for creating stable glasses. These strategies often involve optimizing physical quantities, such as local packing properties and hyperuniformity of density fluctuations, to achieve increased glass stability.

**Numerical Model and Bulk Properties**

A two-dimensional model of hard disks is used to study the bulk properties of glasses. The model consists of a binary distribution of diameters with a diameter ratio of 1:1.4 and composition 65:35. The authors use conventional Monte Carlo methods to simulate the system.

**Design Strategies for Stable Glasses**

The document discusses several design strategies for creating stable glasses, including:

1. **Coupled Position-Diameter Dynamics**: This strategy involves removing thermal fluctuations entirely and using a gradient descent algorithm with evolving diameters to